**par sensitivity Analysis**  

In [1]:
from oreABBR import *; import ORE as ore; import oreUTIL as ou
params  = ore.Parameters()   ; params.fromFile('Input/oreP.xml')
mainOBJ = ore.OREApp(params) ; mainOBJ.run()
ou.chkERR(mainOBJ)
print('asof:',mainOBJ.getInputs().asof()); mainOBJ.getReportNames()

Run time:0.11 sec, Errors:0
asof: August 19th, 2022


('dividends',
 'fixings',
 'jacobi',
 'jacobi_inverse',
 'marketdata',
 'par_sensitivity',
 'pricingstats',
 'runtimes',
 'scenario_par_rates',
 'sensitivity',
 'sensitivity_config',
 'sensitivity_scenario',
 'todaysmarketcalibration',
 'todaysmarketcalibration_cashflows')

In [2]:
psnOBJ = mainOBJ.getReport('par_sensitivity')
dfPSN  = ou.rOBJxdf(psnOBJ); print('dfPSN:行列=',dfPSN.shape)

colLST = ['TradeId','Factor_1','Currency','Base NPV','ShiftSize_1','Delta','Gamma']
dfDSP(dfPSN[colLST], fm=fmO)
print( f'Total Delta:{(dfPSN.Delta).sum():,.2f}   '
       f'Total Gamma:{(dfPSN.Gamma).sum():,.2f}')

dfPSN:行列= (10, 14)


,TradeId,Factor_1,Currency,Base NPV,ShiftSize_1,Delta,Gamma
0,OIS-2y,DiscountCurve/JPY/0/1D,JPY,"-230,413.27",0.0100%,0.19,nan
1,OIS-2y,DiscountCurve/JPY/1/6M,JPY,"-230,413.27",0.0100%,0.07,nan
2,OIS-2y,DiscountCurve/JPY/2/1Y,JPY,"-230,413.27",0.0100%,11.79,nan
3,OIS-2y,DiscountCurve/JPY/3/2Y,JPY,"-230,413.27",0.0100%,22.26,nan
4,OIS-2y,DiscountCurve/JPY/4/3Y,JPY,"-230,413.27",0.0100%,0.36,nan
5,OIS-2y,IndexCurve/JPY-TONAR/0/1D,JPY,"-230,413.27",0.0100%,0.00,nan
6,OIS-2y,IndexCurve/JPY-TONAR/1/6M,JPY,"-230,413.27",0.0100%,0.00,nan
7,OIS-2y,IndexCurve/JPY-TONAR/2/1Y,JPY,"-230,413.27",0.0100%,-0.25,nan
8,OIS-2y,IndexCurve/JPY-TONAR/3/2Y,JPY,"-230,413.27",0.0100%,"2,002.52",nan
9,OIS-2y,IndexCurve/JPY-TONAR/4/3Y,JPY,"-230,413.27",0.0100%,-0.00,nan


Total Delta:2,036.93   Total Gamma:0.00


**jacobiレポート**  

In [3]:
# dfJo: Jacobi original,    clSTR: column string
jacOBJ = mainOBJ.getReport('jacobi')
dfJo   = ou.rOBJxdf(jacOBJ); print('dfJo:行列=',dfJo.shape)
print(dfJo.iloc[:8]);print(dfJo.iloc[-3:])
clSTR  = ['ParFactor','RawFactor','ParSensitivity',]

dfJo:行列= (38, 3)
             ParFactor            RawFactor  ParSensitivity
0  DiscountCurve/JPY/0  DiscountCurve/JPY/0        0.329690
1  DiscountCurve/JPY/0  DiscountCurve/JPY/1        0.670309
2  DiscountCurve/JPY/1  DiscountCurve/JPY/0       -0.005317
3  DiscountCurve/JPY/1  DiscountCurve/JPY/1        0.950792
4  DiscountCurve/JPY/1  DiscountCurve/JPY/2        0.054499
5  DiscountCurve/JPY/2  DiscountCurve/JPY/0       -0.002695
6  DiscountCurve/JPY/2  DiscountCurve/JPY/1       -0.008265
7  DiscountCurve/JPY/2  DiscountCurve/JPY/2        0.989247
                 ParFactor               RawFactor  ParSensitivity
35  IndexCurve/JPY-TONAR/4  IndexCurve/JPY-TONAR/2        0.000243
36  IndexCurve/JPY-TONAR/4  IndexCurve/JPY-TONAR/3       -0.010448
37  IndexCurve/JPY-TONAR/4  IndexCurve/JPY-TONAR/4        1.014172


**5×5ヤコビ行列**  
- CSVに省略されている要素はゼロ

In [4]:
# curve string        curve ID          Node list        Jacobi matrix
cvSTR,                  cvID,             ndLST,             dfJM       =\
['Discount','Index'],['dc','ix'],['1D','6M','1Y','2Y','3Y'],  {}
for ss,ii in zip(cvSTR,cvID):
	dfJM[ii] = dfJo[dfJo[clSTR[0]].str.startswith(ss)]
	dfJM[ii] = dfJM[ii].pivot(index=clSTR[0], columns=clSTR[1],
                values=clSTR[2]                              ).fillna(0.0)
	dfJM[ii].index  = pd.Index(ndLST,name=clSTR[0])
	dfJM[ii].columns= pd.Index(ndLST,name=clSTR[1])
dfJM

{'dc': RawFactor        1D        6M        1Y        2Y        3Y
 ParFactor                                                  
 1D         0.329690  0.670309  0.000000  0.000000  0.000000
 6M        -0.005317  0.950792  0.054499  0.000000  0.000000
 1Y        -0.002695 -0.008265  0.989247  0.021891  0.000000
 2Y        -0.001346 -0.004129  0.000241  0.989112  0.016432
 3Y        -0.000897 -0.002750  0.000243 -0.010448  1.014172,
 'ix': RawFactor        1D        6M        1Y        2Y        3Y
 ParFactor                                                  
 1D         0.329690  0.670309  0.000000  0.000000  0.000000
 6M        -0.005317  0.950792  0.054499  0.000000  0.000000
 1Y        -0.002695 -0.008265  0.989247  0.021891  0.000000
 2Y        -0.001346 -0.004129  0.000241  0.989112  0.016432
 3Y        -0.000897 -0.002750  0.000243 -0.010448  1.014172}

**raw Deltaの非スパース変換**

In [5]:
#sparse column  value column   Delta dic
spsCL,            valCL,        dfDL        =\
'Factor_1',      'Delta',        {}
snOBJ = mainOBJ.getReport("sensitivity"); dfSN = ou.rOBJxdf(snOBJ)
for ss,ii in zip(cvSTR,cvID):
  dfDL[ii]= dfSN[dfSN[spsCL].str.startswith(ss)]
  ftYR    = dfDL[ii][spsCL].str.rsplit('/',n=1).str[-1]        #factor year
  dfDL[ii]= dfDL[ii].set_index(ftYR)[valCL]
  dfDL[ii]= dfDL[ii].reindex(ndLST, fill_value=0.0)
dfDL

{'dc': Factor_1
 1D     0.000000
 6M     0.000000
 1Y    11.674416
 2Y    22.271279
 3Y     0.735250
 Name: Delta, dtype: float64,
 'ix': Factor_1
 1D      -2.694803
 6M      -8.264059
 1Y       0.234047
 2Y    1980.707041
 3Y      32.902144
 Name: Delta, dtype: float64}

**jacobi_inverseレポート**  
- Index Curve

In [6]:
# dfJIo: Jacobi inverse original
jIvOBJ = mainOBJ.getReport('jacobi_inverse')
dfJIo  = ou.rOBJxdf(jIvOBJ); print('dfJIo:行列=',dfJIo.shape)
print(dfJIo.iloc[:3]);print(dfJIo.iloc[-3:])
clSTR = ['RawFactor(z)','ParFactor(c)','dz/dc',]

dfJIo:行列= (50, 3)
          RawFactor(z)         ParFactor(c)     dz/dc
0  DiscountCurve/JPY/0  DiscountCurve/JPY/0  3.000003
1  DiscountCurve/JPY/1  DiscountCurve/JPY/0  0.016304
2  DiscountCurve/JPY/2  DiscountCurve/JPY/0  0.008219
              RawFactor(z)            ParFactor(c)     dz/dc
47  IndexCurve/JPY-TONAR/2  IndexCurve/JPY-TONAR/4  0.000362
48  IndexCurve/JPY-TONAR/3  IndexCurve/JPY-TONAR/4 -0.016378
49  IndexCurve/JPY-TONAR/4  IndexCurve/JPY-TONAR/4  0.985857


In [7]:
dfJIm = {}                                          # Jim:Jacobi Inv matrix
for ss,ii in zip(cvSTR,cvID):
	dfJIm[ii] = dfJIo[dfJIo[clSTR[0]].str.startswith(ss)]
	dfJIm[ii] = dfJIm[ii].pivot(index=clSTR[1], columns=clSTR[0],
                values=clSTR[2]                              ).fillna(0.0)
	dfJIm[ii].index  = pd.Index(ndLST,name=clSTR[1])
	dfJIm[ii].columns= pd.Index(ndLST,name=clSTR[0])
dfJIm

{'dc': RawFactor(z)        1D        6M        1Y        2Y        3Y
 ParFactor(c)                                                  
 1D            3.000003  0.016304  0.008219  0.004104  0.002737
 6M           -2.114003  1.039768  0.002896  0.001446  0.000964
 1Y            0.116465 -0.057283  1.010715 -0.000322 -0.000298
 2Y           -0.002577  0.001268 -0.022365  1.010842  0.010420
 3Y            0.000042 -0.000021  0.000362 -0.016378  0.985857,
 'ix': RawFactor(z)        1D        6M        1Y        2Y        3Y
 ParFactor(c)                                                  
 1D            3.000003  0.016304  0.008219  0.004104  0.002737
 6M           -2.114003  1.039768  0.002896  0.001446  0.000964
 1Y            0.116465 -0.057283  1.010715 -0.000322 -0.000298
 2Y           -0.002577  0.001268 -0.022365  1.010842  0.010420
 3Y            0.000042 -0.000021  0.000362 -0.016378  0.985857}

In [8]:
print('(hc)Par Delta-discount:', dfJIm['dc'].to_numpy() @ dfDL['dc'].to_numpy())
print('(hc)Par Delta-index:'   , dfJIm['ix'].to_numpy() @ dfDL['ix'].to_numpy())

(hc)Par Delta-discount: [ 0.18936  0.06672 11.79211 22.2593   0.36432]
(hc)Par Delta-index: [   0.00158    0.00056   -0.25213 2002.51513   -0.00302]


**全体の10×10ヤコビアン**  

- 今回、curve間の要素はゼロなので、全体は2個の5×5 blockからなるblock diagonal行列

In [9]:
# fctrODR = [f'{cc}/{ii}' for cc in cvNM for ii in range(len(nodLST))]
# Jall = (
#     dfJC.pivot(index='ParFactor', columns='RawFactor', values='ParSensitivity')
#     .reindex(index=fctrODR, columns=fctrODR)
#     .fillna(0.0)
# )
# display(Jall)
# print('shape:', Jall.shape)

**pivot例**
- 列データを表に変形

In [10]:
dfFD = pd.DataFrame({ 'City':  ['Tky', 'Tky', 'Osk'],
                      'Food':  ['Ssi', 'Rmn', 'Ssi'],
                      'Score': [  90,   80,    70  ]})       ;print(dfFD, '\n')
dfPVT= dfFD.pivot(index='City',columns='Food',values='Score');print(dfPVT)

  City Food  Score
0  Tky  Ssi     90
1  Tky  Rmn     80
2  Osk  Ssi     70 

Food   Rmn   Ssi
City            
Osk    NaN  70.0
Tky   80.0  90.0


- reindexで行・列を指定順に揃え、fillna(0)を実行

In [11]:
dfPVT = (dfPVT.reindex(index=['Tky','Osk'],columns=['Ssi','Rmn'])
					    .fillna(0)                                         ); print(dfPVT)

Food   Ssi   Rmn
City            
Tky   90.0  80.0
Osk   70.0   0.0
